# Ejercicio 1 — Análisis Estadístico Descriptivo
## Practica Final: Estadística para Data Science

**Dataset:** Loan Approval Data 2025 (Kaggle)  
**Autor:** Alejandro Pujana Quintero  
**Proposito:** Cuaderno de trabajo paralelo a `ejercicio1_descriptivo.py`.  
Documenta la logica, decisiones tecnicas e interpretacion economica del EDA.

---

### Variable objetivo: `loan_amount`

El monto del prestamo aprobado en USD (continua, rango 500–100,000).  
**Por que `loan_amount` y no `loan_to_income_ratio`?**  
`loan_amount` es la decision de negocio que el banco quiere predecir: dado el perfil del solicitante, cuanto prestarle.  
`loan_to_income_ratio` = loan_amount / annual_income — ya contiene el target en el numerador.  
Usarla como predictor seria data leakage: el modelo veria informacion derivada del target antes de predecirlo.

---

### Columnas excluidas del modelo

| Columna | Razon |
|---|---|
| `customer_id` | Identificador administrativo. Sin valor predictivo — el string 'CUST100042' no dice nada sobre capacidad de pago. |
| `loan_status` | Data leakage. Es el resultado final de la decision de credito. En produccion no existiria al momento de predecir. |
| `payment_to_income_ratio` | Multicolinealidad perfecta (r=1.000) con `loan_to_income_ratio`. XtX singular, OLS no tiene solucion unica. |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Reproducibilidad — requerida por el enunciado en todos los scripts
np.random.seed(42)

# Ruta relativa desde notebooks/ hacia los datos
BASE_DIR  = Path('..') / 'practica_final_Pujana_Quintero_Alejandro'
DATA_PATH = BASE_DIR / 'data' / 'Loan_approval_data_2025.csv'

sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'Memoria: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

## A) Resumen Estructural

### Tipos de dato — significado

- **`int64`**: entero sin decimales. `age`, `annual_income`, `loan_amount`. El 64 indica 64 bits de memoria por valor.
- **`float64`**: decimal. `years_employed` (4.5 anos), `interest_rate` (14.23%). Precision decimal necesaria.
- **`str`**: texto. Variables categoricas. No usables directamente en modelos — requieren One-Hot Encoding.

> `defaults_on_file` (0/1) y `delinquencies_last_2yrs` (0-9) son `int64` pero conceptualmente discretas.  
> Sus coeficientes en regresion se interpretan diferente a los de variables continuas.

In [ ]:
print('Tipos de dato:')
for col, dtype in df.dtypes.items():
    nulos = df[col].isnull().sum()
    print(f'  {col:<35} {str(dtype):<12} nulos: {nulos}')

print(f'\nHallazgo: {df.isnull().sum().sum()} valores nulos en todo el dataset.')
print('Decision: no se requiere imputacion ni eliminacion de filas.')

## B) Estadísticos Descriptivos

### Guia de interpretacion

| Estadistico | Que mide | Cuando usarlo |
|---|---|---|
| Media | Promedio aritmetico | Distribuciones simetricas sin outliers |
| Mediana | Valor central (P50) | Distribuciones asimetricas o con outliers |
| Std | Dispersion alrededor de la media | Comparar variabilidad entre variables |
| IQR | Rango del 50% central | Robusto ante outliers, base para deteccion |
| Skewness | Asimetria (+: cola derecha, -: cola izquierda) | Elegir media vs mediana como representativo |
| Curtosis | Altura del pico (0=normal, >0=leptocurtica) | Detectar distribuciones con outliers extremos |

In [ ]:
DROP_COLS = ['customer_id', 'loan_status', 'payment_to_income_ratio']
df_clean  = df.drop(columns=DROP_COLS)

NUM_COLS = [
    'age', 'years_employed', 'annual_income', 'credit_score',
    'credit_history_years', 'savings_assets', 'current_debt',
    'defaults_on_file', 'delinquencies_last_2yrs', 'derogatory_marks',
    'interest_rate', 'debt_to_income_ratio', 'loan_to_income_ratio', 'loan_amount'
]

stats = pd.DataFrame({
    'media'    : df_clean[NUM_COLS].mean(),
    'mediana'  : df_clean[NUM_COLS].median(),
    'std'      : df_clean[NUM_COLS].std(),
    'Q1'       : df_clean[NUM_COLS].quantile(0.25),
    'Q3'       : df_clean[NUM_COLS].quantile(0.75),
    'IQR'      : df_clean[NUM_COLS].quantile(0.75) - df_clean[NUM_COLS].quantile(0.25),
    'skewness' : df_clean[NUM_COLS].skew(),
    'curtosis' : df_clean[NUM_COLS].kurtosis(),
    'min'      : df_clean[NUM_COLS].min(),
    'max'      : df_clean[NUM_COLS].max(),
}).round(3)

stats

### Interpretacion variable por variable

**`age`** — Media 35.8, mediana 33, skewness +0.77. Mas solicitantes jovenes que mayores.  
Los saltos en el histograma son un artefacto visual: age es un entero discreto y con bins de 40, algunos capturan mas valores enteros que otros. No es un error del dato.

**`annual_income`** — Media $64,151 vs mediana $41,608. Alta asimetria positiva tipica de distribuciones de ingreso. Pocos individuos con ingresos muy altos jalan la media hacia arriba. La mediana es mas representativa del caso tipico.

**`credit_score`** — Casi perfectamente simetrica (skewness ≈ 0). Se comporta como una normal por diseno: los bureaus de credito (FICO) calibran sus modelos para que el score tenga distribucion normal en la poblacion general. Esta simetria indica ausencia de sesgo de seleccion en el dataset.

**`savings_assets`** — Curtosis 57.8 (normal=0). Distribucion con masa enorme en cero (mediana $568) y cola derecha muy larga (max $300,000+). Economicamente real — la distribucion de riqueza es inherentemente desigual. **No eliminar outliers.** Documentar como limitacion del modelo. Una transformacion log(savings_assets + 1) normalizaria la distribucion pero no es requerida por el enunciado.

**`defaults_on_file`** — Binaria (0/1). Mediana 0: el 94.65% no tiene impagos. El histograma con dos barras es correcto, no es un error visual.

**`derogatory_marks`** — 12.79% outliers (valores 2-4). Son perfiles con historial crediticio muy deteriorado (bancarrotas, cuentas a cobro). Economicamente relevantes — no se eliminan.

**`interest_rate`** — Distribucion casi uniforme (7.5–23%). Inusual en datos reales. Sugiere asignacion pseudoaleatoria en el dataset sintetico. Tasas bajas (7.5–10%) corresponden a mejores perfiles crediticios.

**`debt_to_income_ratio`** — Mediana 0.30.  
La deuda acumulada representa el 30% del ingreso anual del solicitante. **No** significa que el 30% de los ingresos 'son deuda' — es relacion stock/flujo. DTI < 0.43 es el umbral tipico de aprobacion hipotecaria.

**`loan_to_income_ratio`** — Mediana 0.60.  
El monto del prestamo equivale al 60% del ingreso anual. **No** significa que el 60% de ingresos 'se van' en el prestamo — eso depende del plazo y la tasa. Si gana $50k y pide $30k (LTI=0.6) a 5 anos, el pago mensual es ~$500/mes = 12% del ingreso mensual, no 60%.

**`loan_amount`** — Skewness +0.93, mediana $26,100 vs media $33,042.  
Prestamos pequenos mas frecuentes, con cola de prestamos grandes. Variables con mayor peso para el monto:  
1. `loan_to_income_ratio` (r=0.66) — el banco calibra el prestamo al ingreso  
2. `annual_income` (r=0.51) — mayor ingreso permite mayor prestamo  
3. `current_debt` (r=0.36) — menor deuda existente permite pedir mas

## C) Distribuciones — Histogramas y Outliers

### IQR vs Z-Score — por que elegimos IQR

**Z-Score** asume distribucion normal. Con skewness alto (savings_assets = +11.4),  
la media y std se distorsionan y el metodo sobredetecta outliers en la cola.

**IQR** es no-parametrico. Usa Q1, Q3 e IQR — estadisticos robustos ante asimetria.  
Eleccion: **IQR** por la asimetria presente en multiples variables.

### Tratamiento: NO eliminar
Los outliers son valores economicamente validos:  
- `savings_assets` altos: individuos con mayor patrimonio  
- `derogatory_marks` altos: perfiles con historial deteriorado (informacion relevante)  

**Trade-off**: mantener outliers puede generar coeficientes menos estables en OLS.  
Mitigacion en Ejercicio 2: StandardScaler para reducir impacto de magnitudes extremas.

In [ ]:
# Tabla de outliers IQR
registros = []
for col in NUM_COLS:
    Q1  = df_clean[col].quantile(0.25)
    Q3  = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    n   = ((df_clean[col] < Q1 - 1.5*IQR) | (df_clean[col] > Q3 + 1.5*IQR)).sum()
    registros.append({
        'variable'    : col,
        'Q1'          : round(Q1, 2),
        'Q3'          : round(Q3, 2),
        'n_outliers'  : n,
        'pct_outliers': round(n / len(df_clean) * 100, 2)
    })

df_out = pd.DataFrame(registros).set_index('variable')
print('Variables con outliers > 5%:')
display(df_out[df_out['pct_outliers'] > 5])
print('\nHallazgo: loan_amount no tiene outliers — todos los montos son prestamos reales validos.')

## Boxplots — importancia en el analisis

Un boxplot comprime 5 estadisticos en una figura:  
**caja** = Q1 a Q3 (50% central) | **linea** = mediana | **bigotes** = 1.5×IQR | **puntos** = outliers

Son fundamentales aqui porque permiten comparar `loan_amount` entre grupos categoricos.

### Hallazgos clave

**`occupation_status`**: Self-Employed mediana mas alta (~$40k). Student mediana baja (~$7k) PERO outliers hasta $100k — estudiantes que reciben prestamos muy altos (Education). Contradice la intuicion simple.

**`product_type`**: Personal Loan mediana mas alta y mayor varianza. Line of Credit concentra montos bajos.

**`loan_intent`**: Medianas similares entre categorias (~$25k-$30k). El proposito del prestamo no determina fuertemente el monto — lo que importa es el perfil financiero.

In [ ]:
CAT_COLS = ['occupation_status', 'product_type', 'loan_intent']
TARGET   = 'loan_amount'

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, cat in enumerate(CAT_COLS):
    order = df_clean.groupby(cat)[TARGET].median().sort_values(ascending=False).index
    sns.boxplot(
        data=df_clean, x=cat, y=TARGET, order=order,
        hue=cat, palette='muted', linewidth=0.8,
        fliersize=2, legend=False, ax=axes[i]
    )
    axes[i].set_title(f'loan_amount por {cat}', fontweight='bold')
    axes[i].tick_params(axis='x', rotation=20)
    axes[i].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.suptitle('Distribucion de loan_amount por Variable Categorica', fontweight='bold')
plt.tight_layout()
plt.show()

## D) Variables Categoricas

**`occupation_status`**: Employed domina (69.9%) — perfil de menor riesgo para el banco.  
Students (9.7%) pocos pero su presencia permite estudiar prestamos educativos.  
Desbalance 7.2x implica que el coeficiente de Student en regresion tendra mayor incertidumbre.

**`product_type`**: Credit Card (44.9%) es el producto mas accesible y de menor monto.  
Lines of Credit son productos mas sofisticados — menos frecuentes.

**`loan_intent`**: Personal (24.9%) domina por ser categoria residual.  
Education segundo lugar — coherente con perfil demografico joven.  
Debt Consolidation menor frecuencia — solo perfiles con deuda significativa que refinancian.

In [ ]:
for cat in CAT_COLS:
    freq = pd.DataFrame({
        'n'             : df_clean[cat].value_counts(),
        'pct'           : df_clean[cat].value_counts(normalize=True).mul(100).round(1),
        'mediana_loan'  : df_clean.groupby(cat)['loan_amount'].median().astype(int),
    })
    print(f'\n{cat}:')
    print(freq.to_string())

## E) Correlaciones de Pearson

### Escala de interpretacion

| |r| | Interpretacion |
|---|---|
| 0.00 – 0.19 | Despreciable |
| 0.20 – 0.39 | Debil |
| 0.40 – 0.59 | Moderada |
| 0.60 – 0.79 | Fuerte |
| 0.80 – 1.00 | Muy fuerte |

### Hallazgos clave

- `loan_to_income_ratio` r=+0.66 con target: el banco presta en proporcion al ingreso
- `annual_income` r=+0.51: mayor ingreso permite mayor prestamo
- `credit_score` vs `interest_rate` r=-0.49: mejores perfiles obtienen menores tasas
- `current_debt` vs `annual_income` r=+0.70: mas ingreso = mas acceso a credito (no mas riesgo)
- `age` vs `years_employed` r=+0.63 y vs `credit_history_years` r=+0.64: consecuencia logica de la edad
- `payment_to_income_ratio` excluida: r=1.000 con `loan_to_income_ratio` — multicolinealidad perfecta

In [ ]:
corr = df_clean[NUM_COLS].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.4, annot_kws={'size': 7},
    square=True, ax=ax,
    cbar_kws={'shrink': 0.8, 'label': 'Correlacion de Pearson'}
)
ax.set_title('Matriz de Correlaciones de Pearson — Variables Numericas', fontweight='bold')
plt.tight_layout()
plt.show()

corr_target = corr['loan_amount'].drop('loan_amount').abs().sort_values(ascending=False)
print('\nTop-5 correlaciones con loan_amount:')
for var, val in corr_target.head(5).items():
    s = corr.loc[var, 'loan_amount']
    print(f'  {var:<35} r = {s:+.4f}')

## Resumen para Respuestas.md

**P1.1**: Dataset Kaggle, target `loan_amount` continua. Sentido: predecir cuanto prestar basado en perfil financiero del solicitante.

**P1.2**: Variables con asimetria positiva: `annual_income`, `years_employed`, `savings_assets`, `current_debt`.  
`credit_score` simetrica (normal por diseno FICO).  
Outliers significativos: `savings_assets` (13.02%) y `derogatory_marks` (12.79%).  
Decision: NO eliminar. Son valores economicamente validos. Documentar como limitacion.

**P1.3**: Top-3 con `loan_amount`: `loan_to_income_ratio` (r=+0.6622), `annual_income` (r=+0.5111), `current_debt` (r=+0.3598).

**P1.4**: 0 valores nulos en todo el dataset. No se requirio tratamiento.